In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are placed in working directory)
metric_file = "metric.csv"
trace_file = "trace.csv"
log_file = "log.csv"
error_file = "error_logs.csv"

# Read files
df_metric = pd.read_csv(metric_file)
df_trace = pd.read_csv(trace_file)
df_log = pd.read_csv(log_file)
df_error = pd.read_csv(error_file)

# Normalize timestamp to timezone Asia/Shanghai for reference (do not filter by time)
# Assume timestamps are epoch seconds in UTC; convert and then to Asia/Shanghai
for df in (df_metric, df_trace, df_log, df_error):
    if 'timestamp' in df.columns:
        df['ts_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Aggregated global thresholds per component-KPI using full series
# Group by cmdb_id and kpi_name and compute required stats
grp = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
agg_basic = grp.agg(count='count', min='min', max='max', mean='mean', std='std').reset_index()

# Percentiles
p50 = grp.quantile(0.50).rename('p50').reset_index()
p90 = grp.quantile(0.90).rename('p90').reset_index()
p95 = grp.quantile(0.95).rename('p95').reset_index()
p99 = grp.quantile(0.99).rename('p99').reset_index()

# Merge percentiles into agg_basic
agg_stats = agg_basic.merge(p50, on=['cmdb_id','kpi_name']) \
                     .merge(p90, on=['cmdb_id','kpi_name']) \
                     .merge(p95, on=['cmdb_id','kpi_name']) \
                     .merge(p99, on=['cmdb_id','kpi_name'])

# Round numeric columns for compact display
num_cols = ['min','max','mean','std','p50','p90','p95','p99']
agg_stats[num_cols] = agg_stats[num_cols].round(6)

# Sort by count desc and limit to top 50 groups
agg_stats_top50 = agg_stats.sort_values('count', ascending=False).head(50).reset_index(drop=True)

# 2) List of unique cmdb_id values present across metric/trace/log/error_logs (compact)
cmdb_metric = set(df_metric['cmdb_id'].dropna().unique())
cmdb_trace = set(df_trace['cmdb_id'].dropna().unique())
cmdb_log = set(df_log['cmdb_id'].dropna().unique())
cmdb_error = set(df_error['cmdb_id'].dropna().unique())

cmdb_union = sorted(cmdb_metric | cmdb_trace | cmdb_log | cmdb_error)
unique_cmdbs = pd.DataFrame({'cmdb_id': cmdb_union})

# 3) Top 20 most frequent trace_name values with counts from trace.csv
top_trace = df_trace['trace_name'].value_counts().head(20).rename_axis('trace_name').reset_index(name='count')

# 4) Top 20 most frequent kpi_name values with counts from metric.csv
top_kpi = df_metric['kpi_name'].value_counts().head(20).rename_axis('kpi_name').reset_index(name='count')

# 5) Top 20 most frequent log_name values with counts from log.csv
top_log = df_log['log_name'].value_counts().head(20).rename_axis('log_name').reset_index(name='count')

# Display compact outputs (variables returned)
agg_stats_top50, unique_cmdbs, top_trace, top_kpi, top_log

```
Out[1]:
```
# Compose a concise plain-English summary using previously computed variables
total_groups = agg_stats.shape[0]
top50_count = agg_stats_top50.shape[0]

# Top 5 groups (component - kpi) with counts
top5_groups_rows = agg_stats_top50[['cmdb_id','kpi_name','count']].head(5).to_dict(orient='records')
top5_groups = "; ".join([f"{r['cmdb_id']} - {r['kpi_name']} (count={r['count']})" for r in top5_groups_rows])

# Unique cmdb_ids across all files
cmdb_list = unique_cmdbs['cmdb_id'].tolist()
cmdb_compact = ", ".join(cmdb_list)

# Top 5 trace names and top 5 kpi names
top5_trace = top_trace.head(5).apply(lambda r: f"{r['trace_name']} ({r['count']})", axis=1).tolist()
top5_kpi = top_kpi.head(5).apply(lambda r: f"{r['kpi_name']} ({r['count']})", axis=1).tolist()

# Log names present?
log_empty = top_log.shape[0] == 0

summary = (
    f"Summary of telemetry aggregation results:\n\n"
    f"1) Metric component-KPI global thresholds:\n"
    f"   - Total distinct component-KPI groups analyzed: {total_groups}\n"
    f"   - Top returned groups (limited to top {top50_count} by count):\n"
    f"     {top5_groups}\n\n"
    f"2) Unique cmdb_id values across metric/trace/log/error_logs ({len(cmdb_list)} total):\n"
    f"   {cmdb_compact}\n\n"
    f"3) Top trace_name values (top 5 shown):\n"
    f"   - " + "; ".join(top5_trace) + "\n\n"
    f"4) Top kpi_name values (top 5 shown):\n"
    f"   - " + "; ".join(top5_kpi) + "\n\n"
    f"5) Log names:\n"
    f"   - {'No log_name entries found in log.csv.' if log_empty else 'Some log_name entries present (see top_log variable).'}\n\n"
    f"Notes: Full top-20 lists for trace_name and kpi_name are available in variables `top_trace` and `top_kpi`. "
    f"The detailed aggregated statistics (count, min, max, mean, std, p50/p90/p95/p99) for the top 50 groups are in `agg_stats_top50`."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count           min           max          mean           std           p50           p90           p95           p99
0   apache01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  6.061139e+06  6.062959e+06  6.062049e+06    553.602371  6.062043e+06  6.062779e+06  6.062869e+06  6.062941e+06
1    Mysql02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  0.000000e+00  0.000000e+00  0.000000e+00      0.000000  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
2       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2.040136e+09  2.040136e+09  2.040136e+09      0.000000  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09
3       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  8.600000e+01  9.400000e+01  8.890323e+01      2.385440  8.900000e+01  9.200000e+01  9.300000e+01  9.400000e+01
4   apache01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31  4.000000e+00  4.000000e+00  4.000000e+00      0.000000  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00
..       ...                                                ...    ...           ...           ...           ...           ...           ...           ...           ...           ...
45      IG02                         OSLinux-CPU_CPU_CPUSysTime     31  2.078000e-01  5.692210e-01  3.426480e-01      0.094484  3.782000e-01  4.321000e-01  4.509200e-01  5.393470e-01
46  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  7.010062e+07  7.041870e+07  7.026081e+07  85181.045579  7.026210e+07  7.037867e+07  7.039159e+07  7.041225e+07
47  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  1.191600e+05  1.221020e+05  1.205582e+05    856.758414  1.204700e+05  1.217470e+05  1.219205e+05  1.220639e+05
48  apache01                            OSLinux-CPU_CPU_CPULoad     31  1.000000e-02  1.590000e+00  1.680650e-01      0.319264  5.000000e-02  4.200000e-01  6.800000e-01  1.329000e+00
49  apache02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  0.000000e+00  0.000000e+00  0.000000e+00      0.000000  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00

[50 rows x 11 columns],      cmdb_id
0       IG01
1       IG02
2       MG01
3       MG02
4    Mysql01
..       ...
24  apache02
25  dockerA1
26  dockerA2
27  dockerB1
28  dockerB2

[29 rows x 1 columns],                        trace_name  count
0        trace.self.duration_mean    360
1         trace.self.duration_p95    360
2            trace.self.row_count    360
3         trace.to_MG02.row_count    220
4     trace.to_MG02.duration_mean    220
5      trace.to_MG02.duration_p95    220
6     trace.to_MG01.duration_mean    219
7         trace.to_MG01.row_count    219
8      trace.to_MG01.duration_p95    219
9    trace.from_IG02.duration_p95    120
10  trace.from_IG02.duration_mean    120
11      trace.from_IG01.row_count    120
12  trace.from_IG01.duration_mean    120
13      trace.from_IG02.row_count    120
14   trace.from_IG01.duration_p95    120
15      trace.from_MG01.row_count    100
16      trace.from_MG02.row_count    100
17   trace.from_MG02.duration_p95    100
18   trace.from_MG01.duration_p95    100
19  trace.from_MG01.duration_mean    100,                                              kpi_name  count
0      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    374
1          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    374
2                  OSLinux-OSLinux_ZABBIX_Host_Uptime    374
3       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    374
4        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    373
5                             OSLinux-CPU_CPU_CPULoad    371
6                         OSLinux-CPU_CPU_CPUidleutil    371
7                         OSLinux-CPU_CPU_CPUUserTime    371
8                              OSLinux-CPU_CPU_CPUWio    371
9                          OSLinux-CPU_CPU_CPUSysTime    371
10  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    370
11                                             app.rr    341
12                                             app.sr    341
13                                            app.cnt    341
14                                            app.mrt    341
15              OSLinux-OSLinux_MEMORY_MEMORY_UserMem    298
16          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem    298
17           OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem    296
18             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem    296
19       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc    295, Empty DataFrame
Columns: [log_name, count]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure timezone-aware ts_dt in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'ts_dt' not in df_metric.columns or df_metric['ts_dt'].isnull().all():
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global p95 per component-KPI using the full available series (no time filtering)
p95_df = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'] \
    .quantile(0.95).reset_index().rename(columns={'value': 'p95_value'})

# 2) Filter metric.csv to the time window March 4, 2021 21:00 to 21:30 (UTC+8)
start = pd.Timestamp('2021-03-04 21:00:00', tz=tz)
end = pd.Timestamp('2021-03-04 21:30:00', tz=tz)
mask_window = (df_metric['ts_dt'] >= start) & (df_metric['ts_dt'] <= end)
df_win = df_metric.loc[mask_window].copy()

# 3) Merge global p95 into windowed rows and mark anomalies where value >= p95
df_win = df_win.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')
df_win['is_anomaly'] = False
mask_has_p95 = df_win['p95_value'].notna()
df_win.loc[mask_has_p95, 'is_anomaly'] = df_win.loc[mask_has_p95, 'value'] >= df_win.loc[mask_has_p95, 'p95_value']

# 4) Aggregate per component-KPI for groups present in the window
agg_counts = df_win.groupby(['cmdb_id', 'kpi_name']).agg(
    number_of_rows_in_window = ('value', 'count'),
    number_of_anomalies = ('is_anomaly', 'sum'),
    p95_value = ('p95_value', 'first'),
    max_value_in_window = ('value', 'max')
).reset_index()

# earliest anomaly timestamp per group (if any)
earliest = df_win[df_win['is_anomaly']].groupby(['cmdb_id','kpi_name'])['ts_dt'].min().reset_index().rename(columns={'ts_dt':'earliest_anomaly_ts'})

# merge earliest into agg_counts
result_df_all = agg_counts.merge(earliest, on=['cmdb_id','kpi_name'], how='left')

# compute max_excess_ratio with NaN if p95_value is NaN or zero
result_df_all['max_excess_ratio'] = np.where(
    result_df_all['p95_value'].isna() | (result_df_all['p95_value'] == 0),
    np.nan,
    (result_df_all['max_value_in_window'] - result_df_all['p95_value']) / result_df_all['p95_value']
)

# format earliest_anomaly_timestamp as ISO string or None
result_df_all['earliest_anomaly_timestamp'] = result_df_all['earliest_anomaly_ts'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
result_df_all = result_df_all.drop(columns=['earliest_anomaly_ts'])

# Round numeric columns for compactness
for c in ['p95_value','max_value_in_window','max_excess_ratio']:
    if c in result_df_all.columns:
        result_df_all[c] = result_df_all[c].round(6)

# Keep only groups with number_of_rows_in_window > 0 (df_win ensures this)
# Sort by number_of_anomalies desc then max_excess_ratio desc, limit to top 20
result_df_top20 = result_df_all.sort_values(['number_of_anomalies','max_excess_ratio'], ascending=[False, False]).head(20).reset_index(drop=True)

# Total groups with anomaly (number_of_anomalies > 0)
total_groups_with_anomaly = int((result_df_all['number_of_anomalies'] > 0).sum())

# Return compact outputs (stored in kernel variables for follow-up)
result_df_top20, total_groups_with_anomaly

```
Out[3]:
```
# Compose a concise plain-English summary using kernel variables from previous analysis
try:
    groups_in_window = int(result_df_all.shape[0])
    top20 = result_df_top20
    total_with_anom = int(total_groups_with_anomaly)
except NameError:
    summary = "Required analysis variables (result_df_all, result_df_top20, total_groups_with_anomaly) not found in kernel."
    summary
else:
    # Prepare top 3 example rows from the top-20 result for illustration
    examples = []
    for _, r in top20.head(3).iterrows():
        examples.append(
            f"{r['cmdb_id']} - {r['kpi_name']} (rows={int(r['number_of_rows_in_window'])}, "
            f"anomalies={int(r['number_of_anomalies'])}, p95={r['p95_value']}, "
            f"max={r['max_value_in_window']}, excess_ratio={r['max_excess_ratio']}, "
            f"earliest={r['earliest_anomaly_timestamp']})"
        )
    examples_text = "; ".join(examples)

    summary = (
        f"Summary of anomaly detection in window 2021-03-04 21:00–21:30 (UTC+8):\n"
        f"- Component-KPI groups present in the window: {groups_in_window}\n"
        f"- Groups with at least one anomaly (value >= global p95): {total_with_anom}\n"
        f"- Top examples (from the top-20 sorted results): {examples_text}\n\n"
        f"Notes: Many top groups show all rows in the 31-minute window flagged as anomalies (e.g., anomalies=31), "
        f"often because the window values equal the group's global p95 (resulting in zero excess ratio). "
        f"Full per-group results are available in variables `result_df_all` (all groups in window) and `result_df_top20` (top 20 summary)."
    )
    summary

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                           kpi_name  number_of_rows_in_window  number_of_anomalies     p95_value  max_value_in_window  max_excess_ratio earliest_anomaly_timestamp
0      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                        31                   31  2.040136e+09         2.040136e+09               0.0  2021-03-04T21:00:00+08:00
1      IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                        31                   31  1.000000e+00         1.000000e+00               0.0  2021-03-04T21:00:00+08:00
2      IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                        31                   31  3.000000e-01         3.000000e-01               0.0  2021-03-04T21:00:00+08:00
3      IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                        31                   31  3.000000e+00         3.000000e+00               0.0  2021-03-04T21:00:00+08:00
4      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                        31                   31  2.040136e+09         2.040136e+09               0.0  2021-03-04T21:00:00+08:00
5      IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                        31                   31  8.239351e+08         8.239351e+08               0.0  2021-03-04T21:00:00+08:00
6      IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                        31                   31  1.000000e+00         1.000000e+00               0.0  2021-03-04T21:00:00+08:00
7      IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                        31                   31  3.000000e+00         3.000000e+00               0.0  2021-03-04T21:00:00+08:00
8      MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax                        31                   31  2.040136e+09         2.040136e+09               0.0  2021-03-04T21:00:00+08:00
9      MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed                        31                   31  5.060956e+08         5.060956e+08               0.0  2021-03-04T21:00:00+08:00
10     MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax                        31                   31  2.040136e+09         2.040136e+09               0.0  2021-03-04T21:00:00+08:00
11     MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed                        31                   31  9.468300e+08         9.468300e+08               0.0  2021-03-04T21:00:00+08:00
12  Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                        31                   31  7.000000e+00         7.000000e+00               0.0  2021-03-04T21:00:00+08:00
13  Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                        31                   31  4.000000e+00         4.000000e+00               0.0  2021-03-04T21:00:00+08:00
14  Redis01        redis-Redis_6379_Redis  (connected_clients)                        31                   31  6.020000e+02         6.020000e+02               0.0  2021-03-04T21:00:00+08:00
15  Redis01             redis-Redis_6379_Redis  (expired_keys)                        31                   31  6.000000e+00         6.000000e+00               0.0  2021-03-04T21:00:00+08:00
16  Redis01          redis-Redis_6379_Redis  (keyspace_misses)                        31                   31  6.938000e+03         6.938000e+03               0.0  2021-03-04T21:00:00+08:00
17  Redis01                redis-Redis_6379_Redis  (maxmemory)                        31                   31  4.294967e+09         4.294967e+09               0.0  2021-03-04T21:00:00+08:00
18  Redis01         redis-Redis_6379_Redis  (used_memory_peak)                        31                   31  3.487778e+07         3.487778e+07               0.0  2021-03-04T21:00:00+08:00
19  Redis01                      redis-Redis_6379_redis server                        31                   31  1.000000e+00         1.000000e+00               0.0  2021-03-04T21:00:00+08:00, 1691)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse df_trace if present, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure timezone-aware ts_dt in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'ts_dt' not in df_trace.columns or df_trace['ts_dt'].isnull().all():
    df_trace['ts_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global p95 per (cmdb_id, trace_name) using the full series (global threshold BEFORE filtering)
p95_trace = df_trace.groupby(['cmdb_id', 'trace_name'])['value'] \
    .quantile(0.95).reset_index().rename(columns={'value': 'p95_value'})

# 2) Filter trace to the time window March 4, 2021 21:00 to 21:30 (UTC+8)
start = pd.Timestamp('2021-03-04 21:00:00', tz=tz)
end = pd.Timestamp('2021-03-04 21:30:00', tz=tz)
mask_window = (df_trace['ts_dt'] >= start) & (df_trace['ts_dt'] <= end)
df_trace_win = df_trace.loc[mask_window].copy()

# 3) Merge p95 into windowed rows and mark anomalies where value >= p95_value
df_trace_win = df_trace_win.merge(p95_trace, on=['cmdb_id', 'trace_name'], how='left')
df_trace_win['is_anomaly'] = False
mask_has_p95 = df_trace_win['p95_value'].notna()
df_trace_win.loc[mask_has_p95, 'is_anomaly'] = df_trace_win.loc[mask_has_p95, 'value'] >= df_trace_win.loc[mask_has_p95, 'p95_value']

# 4) Aggregate per (cmdb_id, trace_name) present in the window
agg_trace = df_trace_win.groupby(['cmdb_id', 'trace_name']).agg(
    number_of_rows_in_window = ('value', 'count'),
    number_of_anomalies = ('is_anomaly', 'sum'),
    p95_value = ('p95_value', 'first'),
    max_value_in_window = ('value', 'max')
).reset_index()

# earliest anomaly timestamp per group (if any)
earliest_anom = df_trace_win[df_trace_win['is_anomaly']].groupby(['cmdb_id','trace_name'])['ts_dt'].min().reset_index().rename(columns={'ts_dt':'earliest_anomaly_ts'})
result_trace_all = agg_trace.merge(earliest_anom, on=['cmdb_id','trace_name'], how='left')

# 5) Compute max_excess_ratio; use NaN if p95_value is NaN or zero
result_trace_all['max_excess_ratio'] = np.where(
    result_trace_all['p95_value'].isna() | (result_trace_all['p95_value'] == 0),
    np.nan,
    (result_trace_all['max_value_in_window'] - result_trace_all['p95_value']) / result_trace_all['p95_value']
)

# Format earliest_anomaly_timestamp as ISO string or None for compact display
result_trace_all['earliest_anomaly_timestamp'] = result_trace_all['earliest_anomaly_ts'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
result_trace_all = result_trace_all.drop(columns=['earliest_anomaly_ts'])

# Round numeric columns for compactness
for c in ['p95_value','max_value_in_window','max_excess_ratio']:
    if c in result_trace_all.columns:
        result_trace_all[c] = result_trace_all[c].round(6)

# Keep only groups with number_of_rows_in_window > 0 (df_trace_win ensures this)
# Sort by number_of_anomalies desc then max_excess_ratio desc, limit to top 20
result_trace_top20 = result_trace_all.sort_values(['number_of_anomalies','max_excess_ratio'], ascending=[False, False]).head(20).reset_index(drop=True)

# Total groups with anomaly (number_of_anomalies > 0)
total_trace_groups_with_anomaly = int((result_trace_all['number_of_anomalies'] > 0).sum())

# Expose compact outputs and keep full tables in variables for follow-up inspection
result_trace_top20, total_trace_groups_with_anomaly

```
Out[4]:
```
# Summarize the trace anomaly detection results in plain English (compact)
try:
    top20 = result_trace_top20  # from previous computation
    total_with_anom = total_trace_groups_with_anomaly
    all_groups_in_window = result_trace_all.shape[0]
except NameError:
    summary = "Required analysis variables (result_trace_top20, result_trace_all, total_trace_groups_with_anomaly) not found in kernel."
else:
    # prepare up to 3 example rows from the top-20 results
    examples = []
    for _, r in top20.head(3).iterrows():
        examples.append(
            f"{r['cmdb_id']} | {r['trace_name']} — rows={int(r['number_of_rows_in_window'])}, "
            f"anomalies={int(r['number_of_anomalies'])}, p95={r['p95_value']}, "
            f"max={r['max_value_in_window']}, excess_ratio={r['max_excess_ratio']}, "
            f"earliest={r['earliest_anomaly_timestamp']}"
        )
    examples_text = " ; ".join(examples)
    summary = (
        f"Trace anomaly summary for window 2021-03-04 21:00–21:30 (UTC+8):\n"
        f"- Total distinct (cmdb_id, trace_name) groups present in window: {all_groups_in_window}\n"
        f"- Groups with >=1 anomaly (value >= global p95): {int(total_with_anom)}\n"
        f"- Top examples (from top-20 sorted by anomalies then excess ratio): {examples_text}\n\n"
        f"Notes: Full per-group summary table (top 20) is in `result_trace_top20` and the complete windowed aggregation is in `result_trace_all`."
    )

summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                         trace_name  number_of_rows_in_window  number_of_anomalies  p95_value  max_value_in_window  max_excess_ratio earliest_anomaly_timestamp
0   dockerB2            trace.self.duration_p95                        30                    6   0.014000              0.01400          0.000000  2021-03-04T21:24:00+08:00
1       MG01        trace.to_dockerB1.row_count                        25                    3  58.000000             67.00000          0.155172  2021-03-04T21:19:00+08:00
2   dockerB1          trace.from_MG01.row_count                        25                    3  58.000000             67.00000          0.155172  2021-03-04T21:19:00+08:00
3       MG01      trace.from_dockerA2.row_count                        25                    3  13.000000             15.00000          0.153846  2021-03-04T21:27:00+08:00
4   dockerA2            trace.to_MG01.row_count                        25                    3  13.000000             15.00000          0.153846  2021-03-04T21:27:00+08:00
5       IG01        trace.to_Tomcat02.row_count                        30                    3  64.000000             71.00000          0.109375  2021-03-04T21:03:00+08:00
6   Tomcat02          trace.from_IG01.row_count                        30                    3  64.000000             71.00000          0.109375  2021-03-04T21:03:00+08:00
7       MG02        trace.to_dockerB2.row_count                        25                    3  51.000000             56.00000          0.098039  2021-03-04T21:00:00+08:00
8   dockerB2          trace.from_MG02.row_count                        25                    3  51.000000             56.00000          0.098039  2021-03-04T21:00:00+08:00
9       MG02      trace.from_Tomcat02.row_count                        30                    3  65.000000             70.00000          0.076923  2021-03-04T21:01:00+08:00
10  Tomcat02            trace.to_MG02.row_count                        30                    3  65.000000             70.00000          0.076923  2021-03-04T21:01:00+08:00
11      MG01      trace.from_Tomcat02.row_count                        30                    3  68.000000             72.00000          0.058824  2021-03-04T21:05:00+08:00
12  Tomcat02            trace.to_MG01.row_count                        30                    3  68.000000             72.00000          0.058824  2021-03-04T21:05:00+08:00
13      MG02      trace.from_Tomcat04.row_count                        30                    3  70.000000             70.00000          0.000000  2021-03-04T21:05:00+08:00
14  Tomcat04            trace.to_MG02.row_count                        30                    3  70.000000             70.00000          0.000000  2021-03-04T21:05:00+08:00
15      MG02   trace.from_dockerB1.duration_p95                        25                    2   0.025550              0.71610         27.027397  2021-03-04T21:20:00+08:00
16  dockerB1         trace.to_MG02.duration_p95                        25                    2   0.025550              0.71610         27.027397  2021-03-04T21:20:00+08:00
17      MG02  trace.from_dockerA2.duration_mean                        25                    2   0.039905              0.52850         12.243954  2021-03-04T21:12:00+08:00
18  dockerA2        trace.to_MG02.duration_mean                        25                    2   0.039905              0.52850         12.243954  2021-03-04T21:12:00+08:00
19      MG02   trace.from_dockerA2.duration_p95                        25                    2   0.099860              0.99335          8.947426  2021-03-04T21:12:00+08:00, 258)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse df_metric if available, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure timezone-aware datetime in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'ts_dt' not in df_metric.columns or df_metric['ts_dt'].isnull().all():
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Components of interest inferred from trace anomalies
components = ['dockerA2','dockerB1','MG02','MG01','dockerB2','IG01','Tomcat02','Tomcat04']

# 1) Compute global p95 per (cmdb_id, kpi_name) using the full available series (global thresholds BEFORE filtering)
p95_metric = df_metric.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'p95_value'})

# 2) Filter metrics to the time window March 4, 2021 21:00 to 21:30 (UTC+8) and restrict to selected components
start = pd.Timestamp('2021-03-04 21:00:00', tz=tz)
end = pd.Timestamp('2021-03-04 21:30:00', tz=tz)
mask_window = (df_metric['ts_dt'] >= start) & (df_metric['ts_dt'] <= end) & (df_metric['cmdb_id'].isin(components))
df_metric_win = df_metric.loc[mask_window].copy()

# 3) Merge global p95 into windowed rows (left join)
df_metric_win = df_metric_win.merge(p95_metric, on=['cmdb_id','kpi_name'], how='left')

# 4) Mark anomalies where value >= p95_value (only when p95 exists)
df_metric_win['is_anomaly'] = False
mask_has_p95 = df_metric_win['p95_value'].notna()
df_metric_win.loc[mask_has_p95, 'is_anomaly'] = df_metric_win.loc[mask_has_p95, 'value'] >= df_metric_win.loc[mask_has_p95, 'p95_value']

# 5) Aggregate per (cmdb_id, kpi_name) present in the filtered set
result_metric_all = df_metric_win.groupby(['cmdb_id','kpi_name']).agg(
    number_of_rows_in_window = ('value','count'),
    number_of_anomalies = ('is_anomaly','sum'),
    p95_value = ('p95_value','first'),
    max_value_in_window = ('value','max')
).reset_index()

# earliest anomaly timestamp per group (if any)
earliest_anom = df_metric_win[df_metric_win['is_anomaly']].groupby(['cmdb_id','kpi_name'])['ts_dt'].min().reset_index().rename(columns={'ts_dt':'earliest_anomaly_ts'})
result_metric_all = result_metric_all.merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')

# 6) Compute max_excess_ratio; use NaN if p95_value is NaN or zero
result_metric_all['max_excess_ratio'] = np.where(
    result_metric_all['p95_value'].isna() | (result_metric_all['p95_value'] == 0),
    np.nan,
    (result_metric_all['max_value_in_window'] - result_metric_all['p95_value']) / result_metric_all['p95_value']
)

# Format earliest anomaly timestamp as ISO string or None for compact display
result_metric_all['earliest_anomaly_timestamp'] = result_metric_all['earliest_anomaly_ts'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
result_metric_all = result_metric_all.drop(columns=['earliest_anomaly_ts'])

# Round numeric columns for compactness
for c in ['p95_value','max_value_in_window','max_excess_ratio']:
    if c in result_metric_all.columns:
        result_metric_all[c] = result_metric_all[c].round(6)

# Keep only groups with number_of_rows_in_window > 0 (df_metric_win ensures this)
# Sort by max_excess_ratio desc, limit to top 20
result_metric_top20 = result_metric_all.sort_values('max_excess_ratio', ascending=False).head(20).reset_index(drop=True)

# Count of groups where max_excess_ratio > 0.5 (treat NaN as not counted)
count_of_groups_with_max_excess_ratio_above_0_5 = int((result_metric_all['max_excess_ratio'] > 0.5).sum())

# Compact outputs (full tables stored in result_metric_all and p95_metric for follow-up)
result_metric_top20, count_of_groups_with_max_excess_ratio_above_0_5

```
Out[5]:
```
# Plain-English summary of the metric analysis for selected components in window 2021-03-04 21:00–21:30 (UTC+8)
try:
    top20 = result_metric_top20
    all_groups = result_metric_all
    count_above_0_5 = count_of_groups_with_max_excess_ratio_above_0_5
except NameError:
    summary = "Required results not found in kernel. Ensure `result_metric_top20`, `result_metric_all`, and `count_of_groups_with_max_excess_ratio_above_0_5` exist."
else:
    # prepare up to 3 illustrative examples from the top-20 sorted by max_excess_ratio
    examples = []
    for _, r in top20.head(3).iterrows():
        examples.append(
            f"{r['cmdb_id']} | {r['kpi_name']} — rows={int(r['number_of_rows_in_window'])}, "
            f"anomalies={int(r['number_of_anomalies'])}, p95={r['p95_value']}, "
            f"max={r['max_value_in_window']}, excess_ratio={r['max_excess_ratio']}, "
            f"earliest={r['earliest_anomaly_timestamp']}"
        )
    examples_text = " ; ".join(examples)
    
    summary = (
        "Metric anomaly summary for components [dockerA2, dockerB1, MG02, MG01, dockerB2, IG01, Tomcat02, Tomcat04]\n"
        "in the window 2021-03-04 21:00–21:30 (UTC+8):\n\n"
        f"- Total distinct component-KPI groups present in this filtered window: {int(all_groups.shape[0])}.\n"
        f"- Top 20 groups (sorted by max_excess_ratio) are stored in `result_metric_top20` (shown compactly); examples: {examples_text}.\n"
        f"- Count of groups where max_excess_ratio > 0.5: {int(count_above_0_5)}.\n\n"
        "Notes: The highest excesses include Tomcat02 thread count (excess_ratio=3.0), Tomcat04 CPU WIO (≈1.63), "
        "and IG01 local disk Bps (≈1.4). Full per-group details are in `result_metric_all` and the top-20 summary in `result_metric_top20`."
    )

summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  number_of_rows_in_window  number_of_anomalies   p95_value  max_value_in_window  max_excess_ratio earliest_anomaly_timestamp
0   Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                        31                    2    0.500000               2.0000          3.000000  2021-03-04T21:20:00+08:00
1   Tomcat04                             OSLinux-CPU_CPU_CPUWio                        21                    2    0.091500               0.2411          1.634973  2021-03-04T21:21:00+08:00
2       IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                        16                    1    3.750000               9.0000          1.400000  2021-03-04T21:30:00+08:00
3       MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps                        11                    1    0.500000               1.0000          1.000000  2021-03-04T21:04:00+08:00
4   Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps                        11                    1    0.500000               1.0000          1.000000  2021-03-04T21:13:00+08:00
5   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                        31                    3    1.000000               2.0000          1.000000  2021-03-04T21:08:00+08:00
6       MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                        11                    1  293.266650             580.5333          0.979541  2021-03-04T21:16:00+08:00
7       MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps                        11                    1    1.750000               3.3167          0.895257  2021-03-04T21:16:00+08:00
8   Tomcat04                    OSLinux-CPU_CPU-1_SingleCpuidle                        10                    1    0.365365               0.6643          0.818182  2021-03-04T21:02:00+08:00
9   Tomcat02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps                        10                    1    0.100815               0.1833          0.818182  2021-03-04T21:29:00+08:00
10      IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps                        10                    1    0.550000               1.0000          0.818182  2021-03-04T21:02:00+08:00
11  Tomcat02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                        10                    1   17.966685              32.6667          0.818182  2021-03-04T21:29:00+08:00
12  Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                        10                    1    0.550000               1.0000          0.818182  2021-03-04T21:15:00+08:00
13      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                        10                    1  233.553315             419.7333          0.797163  2021-03-04T21:19:00+08:00
14      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps                        10                    1    2.493300               4.3833          0.758032  2021-03-04T21:19:00+08:00
15      MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...                        10                    1  247.950000             432.0000          0.742287  2021-03-04T21:20:00+08:00
16  Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                        10                    1   14.650000              25.0000          0.706485  2021-03-04T21:29:00+08:00
17      MG01                    OSLinux-CPU_CPU-3_SingleCpuidle                         9                    1    9.494700              15.8245          0.666667  2021-03-04T21:01:00+08:00
18  Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps                         9                    1    1.800000               3.0000          0.666667  2021-03-04T21:20:00+08:00
19  Tomcat02                    OSLinux-CPU_CPU-0_SingleCpuUtil                        11                    1   38.575700              64.2525          0.665621  2021-03-04T21:21:00+08:00, 25)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_metric and p95_metric if present; otherwise load/compute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

tz = pytz.timezone('Asia/Shanghai')
if 'ts_dt' not in df_metric.columns or df_metric['ts_dt'].isnull().all():
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

try:
    p95_metric
except NameError:
    p95_metric = df_metric.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'p95_value'})

# Parameters
components = ["MG02","Tomcat02"]
start = pd.Timestamp('2021-03-04 21:00:00', tz=tz)
end = pd.Timestamp('2021-03-04 21:30:00', tz=tz)

# Filter window and components
mask_window = (df_metric['ts_dt'] >= start) & (df_metric['ts_dt'] <= end) & (df_metric['cmdb_id'].isin(components))
df_metric_win = df_metric.loc[mask_window].copy()

# Merge global p95 (computed from full series) BEFORE marking anomalies
df_metric_win = df_metric_win.merge(p95_metric, on=['cmdb_id','kpi_name'], how='left')

# Mark anomalies (value >= p95_value) only where p95 exists
df_metric_win['is_anomaly'] = False
mask_has_p95 = df_metric_win['p95_value'].notna()
df_metric_win.loc[mask_has_p95, 'is_anomaly'] = df_metric_win.loc[mask_has_p95, 'value'] >= df_metric_win.loc[mask_has_p95, 'p95_value']

# Ensure ordering for consecutive-run calculations
df_metric_win = df_metric_win.sort_values(['cmdb_id','kpi_name','ts_dt']).reset_index(drop=True)

# Define per-group aggregation including longest consecutive anomaly run and anomaly timestamps preview
def agg_group(g):
    n_rows = int(g.shape[0])
    n_anom = int(g['is_anomaly'].sum())
    p95 = g['p95_value'].iloc[0] if 'p95_value' in g.columns else np.nan
    max_val = g['value'].max()
    # max_excess_ratio: NaN if p95 is NaN or zero
    if pd.isna(p95) or p95 == 0:
        max_excess = np.nan
    else:
        max_excess = (max_val - p95) / p95
    # earliest anomaly timestamp
    if n_anom > 0:
        earliest_ts = g.loc[g['is_anomaly'], 'ts_dt'].min()
        # compute longest consecutive anomaly run in the ordered series
        seq = g['is_anomaly'].values
        max_run = 0
        cur = 0
        for v in seq:
            if v:
                cur += 1
                if cur > max_run:
                    max_run = cur
            else:
                cur = 0
        # anomaly timestamps preview (first up to 10)
        preview_ts = list(g.loc[g['is_anomaly'], 'ts_dt'].sort_values().head(10).dt.strftime('%Y-%m-%dT%H:%M:%S%z').tolist())
    else:
        earliest_ts = pd.NaT
        max_run = 0
        preview_ts = []
    return pd.Series({
        'number_of_rows_in_window': n_rows,
        'total_number_of_anomalies': n_anom,
        'earliest_anomaly_timestamp': earliest_ts,
        'longest_consecutive_anomaly_length': int(max_run),
        'p95_value': p95,
        'max_value_in_window': max_val,
        'max_excess_ratio': max_excess,
        'anomalous_timestamps_preview': preview_ts
    })

# Apply aggregation per (cmdb_id, kpi_name)
if df_metric_win.shape[0] > 0:
    grouped = df_metric_win.groupby(['cmdb_id','kpi_name'])
    result_metric_groups = grouped.apply(agg_group).reset_index()
else:
    # empty result structure
    result_metric_groups = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','number_of_rows_in_window','total_number_of_anomalies',
        'earliest_anomaly_timestamp','longest_consecutive_anomaly_length',
        'p95_value','max_value_in_window','max_excess_ratio','anomalous_timestamps_preview'
    ])

# Format earliest_anomaly_timestamp as ISO string or None for compact display
def fmt_ts(x):
    if pd.isna(x):
        return None
    # x is Timestamp with tz
    return pd.to_datetime(x).isoformat()

if 'earliest_anomaly_timestamp' in result_metric_groups.columns:
    result_metric_groups['earliest_anomaly_timestamp'] = result_metric_groups['earliest_anomaly_timestamp'].apply(fmt_ts)

# Round numeric columns for compact display
for c in ['p95_value','max_value_in_window','max_excess_ratio']:
    if c in result_metric_groups.columns:
        result_metric_groups[c] = result_metric_groups[c].astype(float).round(6)

# Keep only groups with number_of_rows_in_window > 0 (should already be the case)
result_metric_groups = result_metric_groups[result_metric_groups['number_of_rows_in_window'] > 0].copy()

# Sort by max_excess_ratio desc and limit to top 20
result_metric_top20 = result_metric_groups.sort_values('max_excess_ratio', ascending=False).head(20).reset_index(drop=True)

# Compute earliest overall anomaly timestamp among these returned rows (minimum non-null)
valid_earliest_ts = result_metric_top20['earliest_anomaly_timestamp'].dropna()
if not valid_earliest_ts.empty:
    # parse to Timestamp to get min, then format ISO
    earliest_overall = pd.to_datetime(valid_earliest_ts.iloc[0])
    # find true min
    earliest_overall = pd.to_datetime(valid_earliest_ts).min()
    earliest_overall_anomaly_timestamp_among_these_groups = earliest_overall.isoformat()
else:
    earliest_overall_anomaly_timestamp_among_these_groups = None

# Expose compact outputs (full intermediate tables kept in variables)
result_metric_top20, earliest_overall_anomaly_timestamp_among_these_groups

```
Out[6]:
```
summary = (
    "Summary of metric anomaly analysis for MG02 and Tomcat02 (window 2021-03-04 21:00–21:30 UTC+8):\n\n"
    "- Returned groups (with >0 rows): 20 component-KPI rows (see `result_metric_top20` variable).\n"
    "- Earliest overall anomaly timestamp among these groups: 2021-03-04T21:00:00+08:00.\n"
    "- Longest consecutive anomaly runs are short (mostly length=1 for these groups).\n\n"
    "Top examples (sorted by max_excess_ratio):\n"
    "1) Tomcat02 | Tomcat-Threads_7441-..._CurrentThreads — rows=31, anomalies=2, p95=0.5, max=2.0, excess_ratio=3.0, earliest=2021-03-04T21:20:00+08:00\n"
    "2) MG02    | OSLinux..._sdb_DSKTps           — rows=11, anomalies=1, p95=0.5, max=1.0, excess_ratio=1.0, earliest=2021-03-04T21:04:00+08:00\n"
    "3) Tomcat02| OSLinux..._sda_DSKRead          — rows=10, anomalies=1, p95≈17.966685, max≈32.6667, excess_ratio≈0.818182, earliest=2021-03-04T21:29:00+08:00\n"
    "4) Tomcat02| OSLinux..._sda_DSKRTps          — rows=10, anomalies=1, p95≈0.100815, max≈0.1833, excess_ratio≈0.818182, earliest=2021-03-04T21:29:00+08:00\n"
    "5) MG02    | OSLinux..._sda_DSKRead          — rows=10, anomalies=1, p95≈233.553315, max≈419.7333, excess_ratio≈0.797163, earliest=2021-03-04T21:19:00+08:00\n\n"
    "Notes:\n"
    "- Most groups show a single anomaly point in the 31-minute window; only a few have 2 anomalies (e.g., Tomcat-Threads and CPU WIO entries).\n"
    "- The full per-group compact table is stored in `result_metric_top20`; the complete intermediate table is in `result_metric_groups` for follow-up inspection."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  number_of_rows_in_window  total_number_of_anomalies earliest_anomaly_timestamp  longest_consecutive_anomaly_length      p95_value  max_value_in_window  max_excess_ratio                       anomalous_timestamps_preview
0   Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                        31                          2  2021-03-04T21:20:00+08:00                                   1       0.500000               2.0000          3.000000  [2021-03-04T21:20:00+0800, 2021-03-04T21:29:00...
1       MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps                        11                          1  2021-03-04T21:04:00+08:00                                   1       0.500000               1.0000          1.000000                         [2021-03-04T21:04:00+0800]
2   Tomcat02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                        10                          1  2021-03-04T21:29:00+08:00                                   1      17.966685              32.6667          0.818182                         [2021-03-04T21:29:00+0800]
3   Tomcat02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps                        10                          1  2021-03-04T21:29:00+08:00                                   1       0.100815               0.1833          0.818182                         [2021-03-04T21:29:00+0800]
4       MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                        10                          1  2021-03-04T21:19:00+08:00                                   1     233.553315             419.7333          0.797163                         [2021-03-04T21:19:00+0800]
5       MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps                        10                          1  2021-03-04T21:19:00+08:00                                   1       2.493300               4.3833          0.758032                         [2021-03-04T21:19:00+0800]
6       MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...                        10                          1  2021-03-04T21:20:00+08:00                                   1     247.950000             432.0000          0.742287                         [2021-03-04T21:20:00+0800]
7   Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                        10                          1  2021-03-04T21:29:00+08:00                                   1      14.650000              25.0000          0.706485                         [2021-03-04T21:29:00+0800]
8   Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps                         9                          1  2021-03-04T21:20:00+08:00                                   1       1.800000               3.0000          0.666667                         [2021-03-04T21:20:00+0800]
9   Tomcat02                    OSLinux-CPU_CPU-0_SingleCpuUtil                        11                          1  2021-03-04T21:21:00+08:00                                   1      38.575700              64.2525          0.665621                         [2021-03-04T21:21:00+0800]
10  Tomcat02                             OSLinux-CPU_CPU_CPUWio                        21                          2  2021-03-04T21:00:00+08:00                                   1       0.054000               0.0873          0.616667  [2021-03-04T21:00:00+0800, 2021-03-04T21:21:00...
11      MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                        10                          1  2021-03-04T21:09:00+08:00                                   1       3.750000               6.0000          0.600000                         [2021-03-04T21:09:00+0800]
12      MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps                        10                          1  2021-03-04T21:20:00+08:00                                   1       3.200000               5.0000          0.562500                         [2021-03-04T21:20:00+0800]
13  Tomcat02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...                        10                          1  2021-03-04T21:29:00+08:00                                   1      39.500000              53.0000          0.341772                         [2021-03-04T21:29:00+0800]
14      MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...                        10                          1  2021-03-04T21:21:00+08:00                                   1       0.155000               0.2000          0.290323                         [2021-03-04T21:21:00+0800]
15      MG02                    OSLinux-CPU_CPU-1_SingleCpuUtil                        10                          1  2021-03-04T21:15:00+08:00                                   1       3.451600               4.4245          0.281869                         [2021-03-04T21:15:00+0800]
16      MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite                        10                          1  2021-03-04T21:25:00+08:00                                   1      18.746700              23.0667          0.230441                         [2021-03-04T21:25:00+0800]
17      MG02                             OSLinux-CPU_CPU_CPUWio                        21                          2  2021-03-04T21:19:00+08:00                                   1       0.951700               1.1474          0.205632  [2021-03-04T21:19:00+0800, 2021-03-04T21:24:00...
18      MG02    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec                         9                          1  2021-03-04T21:05:00+08:00                                   1  204706.400000          243470.0000          0.189362                         [2021-03-04T21:05:00+0800]
19      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps                        10                          1  2021-03-04T21:25:00+08:00                                   1       1.250830               1.4833          0.185853                         [2021-03-04T21:25:00+0800], '2021-03-04T21:00:00+08:00')```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import numpy as np

# Reuse df_error if present; otherwise load
try:
    df_error
except NameError:
    df_error = pd.read_csv("error_logs.csv")

# Ensure tz-aware timestamp column in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'ts_dt' not in df_error.columns or df_error['ts_dt'].isnull().all():
    df_error['ts_dt'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Parameters
components = ["MG02","Tomcat02","Tomcat04","MG01","dockerA2","dockerB1","dockerB2","IG01"]
start = pd.Timestamp('2021-03-04 21:00:00', tz=tz)
end = pd.Timestamp('2021-03-04 21:30:00', tz=tz)

# Filter to window and components
mask_window = (df_error['ts_dt'] >= start) & (df_error['ts_dt'] <= end) & (df_error['cmdb_id'].isin(components))
filtered_error_logs = df_error.loc[mask_window].copy().sort_values(['cmdb_id','ts_dt']).reset_index(drop=True)

# Shorten messages to 200 chars for previews
filtered_error_logs['message_short'] = filtered_error_logs['message'].fillna('').astype(str).str.slice(0,200)

# Aggregate per component: count, earliest timestamp, up to 10 example messages
def collect_examples(g):
    cnt = int(g.shape[0])
    earliest = g['ts_dt'].min()
    examples = g['message_short'].head(10).tolist()
    return pd.Series({
        'component': g.name,
        'count_of_log_rows_in_window': cnt,
        'earliest_log_timestamp_in_window': earliest.isoformat() if pd.notna(earliest) else None,
        'example_messages': examples
    })

if filtered_error_logs.shape[0] > 0:
    summary_df = filtered_error_logs.groupby('cmdb_id').apply(collect_examples).reset_index(drop=True)
    # Keep only components with >=1 row (groupby ensures this), sort by count desc, limit 20
    summary_df = summary_df.sort_values('count_of_log_rows_in_window', ascending=False).head(20).reset_index(drop=True)
else:
    summary_df = pd.DataFrame(columns=['component','count_of_log_rows_in_window','earliest_log_timestamp_in_window','example_messages'])

# Keyword search (case-insensitive)
keywords = ['disk','io','i/o','errno','read error','write error','timeout','OOM','out of memory','exception','error']
# build regex for simple contains; escape special chars where needed and use word-insensitive search
import re
# create pattern that searches any of the keywords, case-insensitive
pattern = r'(' + r'|'.join([re.escape(k) for k in keywords]) + r')'
mask_kw = filtered_error_logs['message'].fillna('').str.contains(pattern, case=False, regex=True)

keyword_matches = filtered_error_logs.loc[mask_kw, ['cmdb_id','ts_dt','message_short']].copy()
# Format timestamp ISO
if not keyword_matches.empty:
    keyword_matches['ts_iso'] = keyword_matches['ts_dt'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    keyword_matches_preview = keyword_matches[['cmdb_id','ts_iso','message_short']].head(5).reset_index(drop=True)
    any_keyword_matches = True
else:
    keyword_matches_preview = pd.DataFrame(columns=['cmdb_id','ts_iso','message_short'])
    any_keyword_matches = False

# Compact outputs for display; keep full filtered_error_logs for follow-up
summary_df, any_keyword_matches, keyword_matches_preview, filtered_error_logs.shape, filtered_error_logs[:0]

```
Out[7]:
```
summary = (
    "Result of error log check (window 2021-03-04 21:00–21:30 UTC+8) for components "
    "['MG02','Tomcat02','Tomcat04','MG01','dockerA2','dockerB1','dockerB2','IG01']:\n\n"
    "- Total log rows in the window: 0. No component has >=1 log row in this window (summary table is empty).\n"
    "- No log messages matched the keywords (disk, io, i/o, errno, read error, write error, timeout, OOM, out of memory, exception, error).\n\n"
    "Stored variables for follow-up: filtered_error_logs (empty), summary_df (empty), keyword_matches_preview (empty)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [component, count_of_log_rows_in_window, earliest_log_timestamp_in_window, example_messages]
Index: [], False, Empty DataFrame
Columns: [cmdb_id, ts_iso, message_short]
Index: [], (0, 5), Empty DataFrame
Columns: [timestamp, cmdb_id, message, ts_dt, message_short]
Index: [])```
```